<a href="https://colab.research.google.com/github/hrishikeshkhade/Verification/blob/main/PART3Q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y cbmc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  minisat
Suggested packages:
  gdb
The following NEW packages will be installed:
  cbmc minisat
0 upgraded, 2 newly installed, 0 to remove and 38 not upgraded.
Need to get 21.4 MB of archives.
After this operation, 82.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 minisat amd64 1:2.2.1-8build1 [88.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 cbmc amd64 5.95.1-4ubuntu1 [21.3 MB]
Fetched 21.4 MB in 2s (11.6 MB/s)
Selecting previously unselected package minisat.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../minisat_1%3a2.2.1-8bu

In [2]:
%%writefile q3p1.c
#define len 4
#include <stdbool.h>

_Bool nondet_bool();
unsigned int nondet_uint();
int nondet_int();

int main() {
   int i,j,k,x;
   int myarr[len];
   unsigned low, high, middle;
   bool exit;

   for(j=0; j<len-1; j++) { __CPROVER_assume(myarr[j] > myarr[j+1]); }

   // ---------- INITIATION: does INV hold right before the loop starts? ----------
   low=0; high=len; exit=0; i=-1;
   __CPROVER_assert( (i>=-1) && (i<len), "INIT: P1 holds before loop" );

   // ---------- CONSECUTION: if INV holds at top of an (arbitrary) iteration
   //            and the loop guard is true, does INV still hold after one
   //            more pass through the body? ----------
   i    = nondet_int();
   low  = nondet_uint();
   high = nondet_uint();
   exit = nondet_bool();
   __CPROVER_assume(low <= len && high <= len && low <= high); // basic domain/typing facts (needed so 'middle' indexes in-bounds)
   __CPROVER_assume( (i>=-1) && (i<len) );                       // assume candidate P1

   if(!exit && low<high) {
      middle = low + ((high-low)>>1);
      if(myarr[middle]<x)      high = middle;
      else if(myarr[middle]>x) low  = middle+1;
      else                     { i = middle; exit = 1; }

      __CPROVER_assert( (i>=-1) && (i<len), "CONSEC: P1 preserved by loop body" );
   }
   return 0;
}

Writing q3p1.c


In [3]:
%%writefile q3p2.c
#define len 4
#include <stdbool.h>

_Bool nondet_bool();
unsigned int nondet_uint();
int nondet_int();

int main() {
   int i,j,k,x;
   int myarr[len];
   unsigned low, high, middle;
   bool exit;

   for(j=0; j<len-1; j++) { __CPROVER_assume(myarr[j] > myarr[j+1]); }

   // ---------- INITIATION ----------
   low=0; high=len; exit=0; i=-1;
   __CPROVER_assert( (i>=0) ? (myarr[i]==x) : 1, "INIT: P2 holds before loop" );

   // ---------- CONSECUTION ----------
   i    = nondet_int();
   low  = nondet_uint();
   high = nondet_uint();
   exit = nondet_bool();
   __CPROVER_assume(low <= len && high <= len && low <= high);
   __CPROVER_assume(i>=-1 && i<len);                 // basic bound on i (P1) -- needed just so myarr[i] indexes safely
   __CPROVER_assume( (i>=0) ? (myarr[i]==x) : 1 );   // assume candidate P2

   if(!exit && low<high) {
      middle = low + ((high-low)>>1);
      if(myarr[middle]<x)      high = middle;
      else if(myarr[middle]>x) low  = middle+1;
      else                     { i = middle; exit = 1; }

      __CPROVER_assert( (i>=0) ? (myarr[i]==x) : 1, "CONSEC: P2 preserved by loop body" );
   }
   return 0;
}

Writing q3p2.c


In [4]:
%%writefile q3p3.c
#define len 4
#include <stdbool.h>

_Bool nondet_bool();
unsigned int nondet_uint();
int nondet_int();

int main() {
   int i,j,k,x;
   int myarr[len];
   unsigned low, high, middle;
   bool exit;

   for(j=0; j<len-1; j++) { __CPROVER_assume(myarr[j] > myarr[j+1]); }
   __CPROVER_assume(k>=0 && k<len);   // arbitrary fixed index, standard trick to encode "for all k"

   // ---------- INITIATION ----------
   low=0; high=len; exit=0; i=-1;
   __CPROVER_assert( (i<0) ? (!(myarr[k]==x)) : 1, "INIT: MISS(P3) holds before loop" );

   // ---------- CONSECUTION ----------
   i    = nondet_int();
   low  = nondet_uint();
   high = nondet_uint();
   exit = nondet_bool();
   __CPROVER_assume(low <= len && high <= len && low <= high);
   __CPROVER_assume(i>=-1 && i<len);
   __CPROVER_assume( (i<0) ? (!(myarr[k]==x)) : 1 );   // assume candidate MISS/P3

   if(!exit && low<high) {
      middle = low + ((high-low)>>1);
      if(myarr[middle]<x)      high = middle;
      else if(myarr[middle]>x) low  = middle+1;
      else                     { i = middle; exit = 1; }

      __CPROVER_assert( (i<0) ? (!(myarr[k]==x)) : 1, "CONSEC: MISS(P3) preserved by loop body" );
   }
   return 0;
}

Writing q3p3.c


In [5]:
!cbmc q3p1.c --unwind 4 --unwinding-assertions

CBMC version 5.95.1 (cbmc-5.95.1) 64-bit x86_64 linux
Parsing q3p1.c
Converting
Type-checking q3p1
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 8 object bits, 56 offset bits (default)
Starting Bounded Model Checking
Unwinding loop main.0 iteration 1 file q3p1.c line 14 function main thread 0
Unwinding loop main.0 iteration 2 file q3p1.c line 14 function main thread 0
Unwinding loop main.0 iteration 3 file q3p1.c line 14 function main thread 0
Runtime Symex: 0.00351818s
size of program expression: 78 steps
simple slicing removed 10 assignments
Generated 2 VCC(s), 1 remaining after simplification
Runtime Postprocess Equation: 4.4886e-05s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00238092s
Running propositional reduction
Post-processing
Runtime Post-process: 9.042e-06s
Solving with MiniSAT 2.2.1 with simplifier
1602 variables, 4061 clauses
SAT 

In [6]:
!cbmc q3p2.c --unwind 4 --unwinding-assertions

CBMC version 5.95.1 (cbmc-5.95.1) 64-bit x86_64 linux
Parsing q3p2.c
Converting
Type-checking q3p2
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 8 object bits, 56 offset bits (default)
Starting Bounded Model Checking
Unwinding loop main.0 iteration 1 file q3p2.c line 14 function main thread 0
Unwinding loop main.0 iteration 2 file q3p2.c line 14 function main thread 0
Unwinding loop main.0 iteration 3 file q3p2.c line 14 function main thread 0
Runtime Symex: 0.00408425s
size of program expression: 79 steps
simple slicing removed 10 assignments
Generated 2 VCC(s), 1 remaining after simplification
Runtime Postprocess Equation: 2.0232e-05s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00427562s
Running propositional reduction
Post-processing
Runtime Post-process: 7.229e-06s
Solving with MiniSAT 2.2.1 with simplifier
2012 variables, 5956 clauses
SAT 

In [7]:
!cbmc q3p3.c --unwind 4 --unwinding-assertions

CBMC version 5.95.1 (cbmc-5.95.1) 64-bit x86_64 linux
Parsing q3p3.c
Converting
Type-checking q3p3
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 8 object bits, 56 offset bits (default)
Starting Bounded Model Checking
Unwinding loop main.0 iteration 1 file q3p3.c line 14 function main thread 0
Unwinding loop main.0 iteration 2 file q3p3.c line 14 function main thread 0
Unwinding loop main.0 iteration 3 file q3p3.c line 14 function main thread 0
Runtime Symex: 0.00428014s
size of program expression: 81 steps
simple slicing removed 10 assignments
Generated 2 VCC(s), 2 remaining after simplification
Runtime Postprocess Equation: 3.3133e-05s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00289045s
Running propositional reduction
Post-processing
Runtime Post-process: 9.82e-06s
Solving with MiniSAT 2.2.1 with simplifier
1878 variables, 5166 clauses
SAT c